# 04 · Smart-Money Analyst

Author: Chris Kim — 2026-05, ChainSense · UCSD DS3

The modeling notebook labels every wallet with an archetype. This notebook narrows to the *smart money* subset — whales and serious DEX traders — pulls their on-chain footprint out of the silver tables, and rolls it up into the compact serving tables the dashboard reads. Built locally on an M4 Mini (8 GB), so every DuckDB step is memory-capped and spills to disk.


## 1. Label wallets, flag smart money

Map each cluster to a human archetype, derive a crude PnL / ROI proxy from in vs. out volume, and flag the smart-money subset: whales and DEX traders with real activity, a confident label, and no volume-accounting blowups.


In [ ]:
import pandas as pd

clusters = pd.read_parquet('../data/gold/wallet_clusters_full.parquet')   # address, cluster, confidence — from notebook 03
features = pd.read_parquet('../data/gold/wallet_features_v4.parquet')      # raw, unstandardized

ARCHETYPE_MAP = {
    0:  'spam_token_sender', 1: 'spam_sniper_bot', 2: 'meme_trader',
    3:  'failed_tx_mev_bot', 4: 'nft_minter', 5: 'dormant_retail',
    6:  'mev_arb_bot', 7: 'casual_defi_explorer', 8: 'whale',
    9:  'long_term_dex_trader', 10: 'single_purpose_user',
    11: 'mainstream_defi_user', 12: 'dex_power_trader',
    13: 'sybil_wallet', 14: 'receive_only_holder', 15: 'light_eth_user',
    16: 'sporadic_user', 17: 'stablecoin_parker', 18: 'nft_trader',
    19: 'dex_aggregator_user', 20: 'power_protocol_user', -1: 'continuum',
}
SMART_MONEY = {'whale', 'long_term_dex_trader', 'dex_power_trader', 'power_protocol_user'}

df = clusters.merge(
    features[['address', 'tx_count', 'in_volume_eth', 'total_volume_eth_out']],
    on='address', how='left')
df['archetype'] = df['cluster'].map(ARCHETYPE_MAP).fillna('unknown')
df['pnl_eth']   = df['in_volume_eth'] - df['total_volume_eth_out']
df['roi_proxy'] = df['pnl_eth'] / df['total_volume_eth_out'].replace(0, 1e-9)
df['is_smart_money'] = (
    df['archetype'].isin(SMART_MONEY)
    & (df['tx_count'] > 50) & (df['tx_count'] < 100_000)   # active, not a contract/bot firehose
    & (df['confidence'] > 0.6)                             # trust the label
    & (df['pnl_eth'].abs() < 50_000)                       # drop volume-accounting blowups
)

out = df[['address', 'archetype', 'confidence', 'tx_count', 'pnl_eth', 'roi_proxy', 'is_smart_money']].copy()
for c, t in {'tx_count': 'int32', 'pnl_eth': 'float32', 'roi_proxy': 'float32',
             'confidence': 'float32', 'is_smart_money': 'bool'}.items():
    out[c] = out[c].astype(t)
out.to_parquet('../analyst_data/wallet_clusters_analyst.parquet', compression='zstd', compression_level=9)

print(f"Smart money: {out['is_smart_money'].sum():,} of {len(out):,} ({out['is_smart_money'].mean():.2%})")

In [ ]:
import pandas as pd
df = pd.read_parquet('../analyst_data/wallet_clusters_analyst.parquet')

counts = df['archetype'].value_counts()
for arch, n in counts.items():
    bar = '#' * int(40 * n / counts.max())
    print(f"  {arch:<22} {n:>9,} ({100 * n / len(df):5.2f}%)  {bar}")
print(f"\nSmart money by archetype:")
print(df[df['is_smart_money']]['archetype'].value_counts().to_string())

## 2. Extract their on-chain footprint

One pass over the silver `transactions` and `token_transfers` tables, keeping any row where smart money sits on either side. The `from_archetype` / `to_archetype` columns are what let the flow tables downstream tell *received-from-noise* apart from *sent-to-noise*.


In [ ]:
import duckdb, os

ANALYST = '../analyst_data/wallet_clusters_analyst.parquet'
SILVER  = '../data/silver/transactions/**/*.parquet'
OUT     = '../analyst_data/smart_money_transactions.parquet'
os.makedirs('../data/duckdb_tmp', exist_ok=True)

con = duckdb.connect()
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")

con.execute(f"""
    CREATE TEMP TABLE smart_money AS
    SELECT address, archetype, confidence
    FROM read_parquet('{ANALYST}') WHERE is_smart_money = TRUE
""")
print(f"Smart money wallets: {con.execute('SELECT COUNT(*) FROM smart_money').fetchone()[0]:,}")

con.execute(f"""
COPY (
    SELECT
        t.tx_hash, t.block_timestamp, t.block_number, t.from_address, t.to_address,
        CAST(t.value_wei AS HUGEINT) / 1e18                                              AS value_eth,
        (CAST(t.gas_used AS HUGEINT) * CAST(t.effective_gas_price_wei AS HUGEINT)) / 1e18 AS gas_spent_eth,
        t.status, t.method_selector,
        sm_from.archetype  AS from_archetype, sm_to.archetype  AS to_archetype,
        sm_from.confidence AS from_confidence, sm_to.confidence AS to_confidence
    FROM read_parquet('{SILVER}', hive_partitioning=true) t
    LEFT JOIN smart_money sm_from ON t.from_address = sm_from.address
    LEFT JOIN smart_money sm_to   ON t.to_address   = sm_to.address
    WHERE sm_from.address IS NOT NULL OR sm_to.address IS NOT NULL
) TO '{OUT}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT}')").fetchone()[0]
print(f"Saved {n:,} transactions ({os.path.getsize(OUT) / 1024**2:.0f} MB)")
con.close()

In [ ]:
import duckdb, os

ANALYST = '../analyst_data/wallet_clusters_analyst.parquet'
SILVER  = '../data/silver/token_transfers/**/*.parquet'
OUT     = '../analyst_data/smart_money_token_transfers.parquet'

con = duckdb.connect()
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")
con.execute("PRAGMA preserve_insertion_order=false")
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")

con.execute(f"""
    CREATE TEMP TABLE smart_money AS
    SELECT address, archetype FROM read_parquet('{ANALYST}') WHERE is_smart_money = TRUE
""")

con.execute(f"""
COPY (
    SELECT
        tt.transaction_hash AS tx_hash, tt.block_timestamp, tt.block_number,
        tt.token_address, tt.from_address, tt.to_address, tt.value AS value_raw,
        sm_from.archetype AS from_archetype, sm_to.archetype AS to_archetype
    FROM read_parquet('{SILVER}', hive_partitioning=true) tt
    LEFT JOIN smart_money sm_from ON tt.from_address = sm_from.address
    LEFT JOIN smart_money sm_to   ON tt.to_address   = sm_to.address
    WHERE sm_from.address IS NOT NULL OR sm_to.address IS NOT NULL
) TO '{OUT}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")

n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT}')").fetchone()[0]
print(f"Saved {n:,} token transfers ({os.path.getsize(OUT) / 1024**2:.0f} MB)")
con.close()

In [ ]:
# Sanity: the join is the riskiest step — confirm it held.
import duckdb
TX = '../analyst_data/smart_money_transactions.parquet'
con = duckdb.connect()

print("Direction + leak check:")
print(con.execute(f"""
    SELECT
        COUNT(*)                                                                       AS n_rows,
        COUNT(*) FILTER (WHERE from_archetype IS NULL AND to_archetype IS NULL)         AS both_null,  -- must be 0
        COUNT(*) FILTER (WHERE from_archetype IS NOT NULL AND to_archetype IS NULL)     AS sender_only,
        COUNT(*) FILTER (WHERE from_archetype IS NULL AND to_archetype IS NOT NULL)     AS receiver_only
    FROM read_parquet('{TX}')
""").fetchdf().to_string(index=False))

print("\nValue / gas scaling (should look like real ETH):")
print(con.execute(f"""
    SELECT ROUND(MEDIAN(value_eth), 4) AS value_median, ROUND(MAX(value_eth), 1) AS value_max,
           ROUND(MEDIAN(gas_spent_eth), 6) AS gas_median
    FROM read_parquet('{TX}')
""").fetchdf().to_string(index=False))

print("\nDate coverage + top methods:")
print(con.execute(f"""
    SELECT MIN(block_timestamp) AS first_ts, MAX(block_timestamp) AS last_ts,
           COUNT(DISTINCT DATE(block_timestamp)) AS days
    FROM read_parquet('{TX}')
""").fetchdf().to_string(index=False))
print(con.execute(f"""
    SELECT method_selector, COUNT(*) AS n
    FROM read_parquet('{TX}') WHERE from_archetype IS NOT NULL
    GROUP BY method_selector ORDER BY n DESC LIMIT 5
""").fetchdf().to_string(index=False))
con.close()

## 3. Roll up into serving tables

The dashboard never touches the multi-GB transaction file. These aggregates are what it actually reads — archetype flows over time and the methods each archetype called. Top-K collapsing keeps the method tables tiny without losing the headline behavior.


In [ ]:
# Hourly ETH flow per archetype
import duckdb, os
RAW = '../analyst_data/smart_money_transactions.parquet'
OUT = '../analyst_data/smart_money_hourly_flow.parquet'

con = duckdb.connect()
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")
con.execute("PRAGMA threads=2"); con.execute("PRAGMA memory_limit='6GB'")
con.execute(f"""
COPY (
    SELECT
        DATE_TRUNC('hour', block_timestamp)        AS hour,
        COALESCE(from_archetype, to_archetype)     AS archetype,
        SUM(CASE WHEN to_archetype   IS NOT NULL THEN value_eth ELSE 0 END)                   AS in_eth,
        SUM(CASE WHEN from_archetype IS NOT NULL THEN value_eth + gas_spent_eth ELSE 0 END)   AS out_eth,
        SUM(CASE WHEN to_archetype   IS NOT NULL THEN value_eth
                 WHEN from_archetype IS NOT NULL THEN -(value_eth + gas_spent_eth) END)       AS net_eth,
        COUNT(DISTINCT CASE WHEN to_archetype IS NOT NULL THEN to_address
                            WHEN from_archetype IS NOT NULL THEN from_address END)            AS n_wallets_active,
        COUNT(*)                                   AS n_txs,
        SUM(CASE WHEN status = 0 THEN 1 ELSE 0 END) AS n_failed_txs
    FROM read_parquet('{RAW}')
    WHERE COALESCE(from_archetype, to_archetype) IS NOT NULL
    GROUP BY hour, archetype ORDER BY hour, archetype
) TO '{OUT}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")
print(f"{con.execute(f'SELECT COUNT(*) FROM read_parquet(\'{OUT}\')').fetchone()[0]:,} rows  ({os.path.getsize(OUT) / 1024**2:.1f} MB)")
con.close()

In [ ]:
# Daily methods per archetype, collapsed to top 10 (rest -> 'other')
import duckdb, os
RAW = '../analyst_data/smart_money_transactions.parquet'
OUT = '../analyst_data/smart_money_daily_methods_top10.parquet'
TOP_K = 10

con = duckdb.connect()
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")
con.execute("PRAGMA threads=2"); con.execute("PRAGMA memory_limit='6GB'")
con.execute(f"""
COPY (
    WITH daily AS (
        SELECT DATE(block_timestamp) AS date, COALESCE(from_archetype, to_archetype) AS archetype,
               method_selector,
               SUM(value_eth) AS volume_eth, SUM(gas_spent_eth) AS gas_eth, COUNT(*) AS n_txs,
               COUNT(DISTINCT CASE WHEN to_archetype IS NOT NULL THEN to_address
                                   WHEN from_archetype IS NOT NULL THEN from_address END) AS n_unique_wallets
        FROM read_parquet('{RAW}')
        WHERE status = 1 AND COALESCE(from_archetype, to_archetype) IS NOT NULL
        GROUP BY date, archetype, method_selector
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY date, archetype ORDER BY n_txs DESC) AS rank FROM daily
    )
    SELECT date, archetype,
           CASE WHEN rank <= {TOP_K} THEN method_selector ELSE 'other' END AS method_selector,
           SUM(volume_eth) AS volume_eth, SUM(gas_eth) AS gas_eth, SUM(n_txs) AS n_txs,
           SUM(n_unique_wallets) AS n_unique_wallets,   -- approximate; pre-grouped sums can overcount
           MIN(rank) AS rank_in_bucket
    FROM ranked
    GROUP BY date, archetype, CASE WHEN rank <= {TOP_K} THEN method_selector ELSE 'other' END
    ORDER BY date, archetype, n_txs DESC
) TO '{OUT}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")
print(f"{con.execute(f'SELECT COUNT(*) FROM read_parquet(\'{OUT}\')').fetchone()[0]:,} rows  ({os.path.getsize(OUT) / 1024**2:.2f} MB)")
con.close()

In [ ]:
# Hourly top-3 methods per archetype, with human-readable signatures
import duckdb, os
RAW = '../analyst_data/smart_money_transactions.parquet'
OUT = '../analyst_data/smart_money_hourly_methods_top3.parquet'
TOP_K = 3

con = duckdb.connect()
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")
con.execute("PRAGMA threads=2"); con.execute("PRAGMA memory_limit='6GB'")

con.execute("""
CREATE TEMP TABLE selector_signatures AS SELECT * FROM (VALUES
    ('0xa9059cbb', 'transfer(address,uint256)'),
    ('0x095ea7b3', 'approve(address,uint256)'),
    ('0x3593564c', 'execute(bytes,bytes[],uint256)'),
    ('0x23b872dd', 'transferFrom(address,address,uint256)'),
    ('0x791ac947', 'swapExactTokensForETHSupportingFeeOnTransferTokens'),
    ('0xb6f9de95', 'swapExactETHForTokensSupportingFeeOnTransferTokens'),
    ('0x7ff36ab5', 'swapExactETHForTokens(uint256,address[],address,uint256)'),
    ('0x12aa3caf', 'swap_1inch_v4'),
    ('0x07ed2379', 'swap_1inch_aggregator'),
    ('0x1249c58b', 'mint()')
) AS t(method_selector, method_signature);
""")

con.execute(f"""
COPY (
    WITH hourly AS (
        SELECT DATE_TRUNC('hour', block_timestamp) AS hour, COALESCE(from_archetype, to_archetype) AS archetype,
               method_selector, SUM(value_eth) AS volume_eth, SUM(gas_spent_eth) AS gas_eth, COUNT(*) AS n_txs
        FROM read_parquet('{RAW}')
        WHERE status = 1 AND COALESCE(from_archetype, to_archetype) IS NOT NULL
        GROUP BY hour, archetype, method_selector
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY hour, archetype ORDER BY n_txs DESC) AS rank FROM hourly
    )
    SELECT r.hour, r.archetype,
           CASE WHEN r.rank <= {TOP_K} THEN r.method_selector ELSE 'other' END AS method_selector,
           CASE WHEN r.rank <= {TOP_K} THEN COALESCE(s.method_signature, 'unknown_' || r.method_selector)
                ELSE 'other' END AS method_signature,
           SUM(r.volume_eth) AS volume_eth, SUM(r.gas_eth) AS gas_eth, SUM(r.n_txs) AS n_txs,
           MIN(r.rank) AS rank_in_bucket
    FROM ranked r LEFT JOIN selector_signatures s ON r.method_selector = s.method_selector
    GROUP BY r.hour, r.archetype,
             CASE WHEN r.rank <= {TOP_K} THEN r.method_selector ELSE 'other' END,
             CASE WHEN r.rank <= {TOP_K} THEN COALESCE(s.method_signature, 'unknown_' || r.method_selector) ELSE 'other' END
    ORDER BY r.hour, r.archetype, n_txs DESC
) TO '{OUT}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")
print(f"{con.execute(f'SELECT COUNT(*) FROM read_parquet(\'{OUT}\')').fetchone()[0]:,} rows  ({os.path.getsize(OUT) / 1024**2:.2f} MB)")
con.close()

In [ ]:
# Hourly token in/out flow per (token, archetype) — the coordination signal
import duckdb, os
RAW = '../analyst_data/smart_money_token_transfers.parquet'
OUT = '../analyst_data/hourly_token_archetype_flow.parquet'

con = duckdb.connect()
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")
con.execute("PRAGMA threads=2"); con.execute("PRAGMA memory_limit='6GB'")
con.execute(f"""
COPY (
    SELECT
        DATE_TRUNC('hour', block_timestamp) AS hour, token_address,
        COALESCE(from_archetype, to_archetype) AS archetype,
        SUM(CASE WHEN to_archetype   IS NOT NULL THEN CAST(value_raw AS DOUBLE) ELSE 0 END) AS tokens_in,
        SUM(CASE WHEN from_archetype IS NOT NULL THEN CAST(value_raw AS DOUBLE) ELSE 0 END) AS tokens_out,
        COUNT(DISTINCT CASE WHEN to_archetype   IS NOT NULL THEN to_address END)   AS distinct_buyers,
        COUNT(DISTINCT CASE WHEN from_archetype IS NOT NULL THEN from_address END) AS distinct_sellers,
        COUNT(*) AS n_transfers
    FROM read_parquet('{RAW}')
    WHERE COALESCE(from_archetype, to_archetype) IS NOT NULL
    GROUP BY hour, token_address, archetype
) TO '{OUT}' (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL 9);
""")
print(f"{con.execute(f'SELECT COUNT(*) FROM read_parquet(\'{OUT}\')').fetchone()[0]:,} rows  ({os.path.getsize(OUT) / 1024**2:.1f} MB)")
con.close()

## 4. Reference tables for USD valuation

Raw token transfers are unit-less integers. Valuing them needs three lookups, pulled once and cached: **decimals** (Alchemy), **prices** (DefiLlama, free batch endpoint), and a curated **CEX address** list to tag exchange flows. Capped to the top 500 tokens by transfer count, which covers the vast majority of volume.


In [ ]:
import duckdb, requests, time, os, pandas as pd

ALCHEMY_KEY = os.environ["ALCHEMY_KEY"]          # never hardcode — load from env
ALCHEMY_URL = f'https://eth-mainnet.g.alchemy.com/v2/{ALCHEMY_KEY}'
TOKEN_TRANSFERS = '../analyst_data/smart_money_token_transfers.parquet'
OUT_DECIMALS, OUT_PRICES, OUT_CEX = (
    '../analyst_data/token_decimals.parquet',
    '../analyst_data/token_prices.parquet',
    '../analyst_data/cex_addresses.parquet')

con = duckdb.connect()
top_tokens = con.execute(f"""
    SELECT token_address FROM read_parquet('{TOKEN_TRANSFERS}')
    GROUP BY token_address ORDER BY COUNT(*) DESC LIMIT 500
""").fetchdf()['token_address'].tolist()

# [1/3] decimals + symbol via Alchemy
rows, errors = [], 0
for i, addr in enumerate(top_tokens):
    try:
        j = requests.post(ALCHEMY_URL, timeout=10, json={
            "jsonrpc": "2.0", "id": 1, "method": "alchemy_getTokenMetadata", "params": [addr]}).json()
        r = {} if 'error' in j else (j.get('result') or {})
        errors += 'error' in j
    except Exception:
        r, errors = {}, errors + 1
    rows.append({'token_address': addr, 'decimals': r.get('decimals'),
                 'symbol': r.get('symbol'), 'name': r.get('name')})
    time.sleep(0.1)
pd.DataFrame(rows).to_parquet(OUT_DECIMALS, compression='zstd')
print(f"decimals: {len(rows)} tokens, {errors} errors")

# [2/3] current prices via DefiLlama (batch up to 100/call)
price_rows = []
for j in range(0, len(top_tokens), 100):
    coins = ','.join(f'ethereum:{a}' for a in top_tokens[j:j + 100])
    try:
        data = requests.get(f'https://coins.llama.fi/prices/current/{coins}', timeout=15).json().get('coins', {})
        price_rows += [{'token_address': k.split(':')[1], 'price_usd': v.get('price'),
                        'symbol': v.get('symbol'), 'timestamp': v.get('timestamp')} for k, v in data.items()]
    except Exception as e:
        print(f"  batch failed: {e}")
    time.sleep(0.3)
pd.DataFrame(price_rows).to_parquet(OUT_PRICES, compression='zstd')
print(f"prices: {len(price_rows)} tokens")

# [3/3] curated CEX hot wallets (verified Etherscan labels)
CEX = {
    '0x3f5ce5fbfe3e9af3971dd833d26ba9b5c936f0be': 'Binance', '0xd551234ae421e3bcba99a0da6d736074f22192ff': 'Binance',
    '0x564286362092d8e7936f0549571a803b203aaced': 'Binance', '0xf977814e90da44bfa03b6295a0616a897441acec': 'Binance',
    '0x71660c4005ba85c37ccec55d0c4493e66fe775d3': 'Coinbase', '0x503828976d22510aad0201ac7ec88293211d23da': 'Coinbase',
    '0xddfabcdc4d8ffc6d5beaf154f18b778f892a0740': 'Coinbase', '0xeb2629a2734e272bcc07bda959863f316f4bd4cf': 'Coinbase',
    '0x2910543af39aba0cd09dbb2d50200b3e800a63d2': 'Kraken', '0xe853c56864a2ebe4576a807d26fdc4a0ada51919': 'Kraken',
    '0x6cc5f688a315f3dc28a7781717a9a798a59fda7b': 'OKX', '0x236f9f97e0e62388479bf9e5ba4889e46b0273c3': 'OKX',
    '0x1151314c646ce4e0efd76d1af4760ae66a9fe30f': 'Bitfinex', '0x5f65f7b609678448494de4c87521cdf6cef1e932': 'Gemini',
}
pd.DataFrame([{'address': a.lower(), 'exchange': n} for a, n in CEX.items()]).to_parquet(OUT_CEX, compression='zstd')
print(f"cex: {len(CEX)} addresses")
con.close()